# B8 · RL moderne : DQN, gradient de politique, PPO et RLHF — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b8_rl_moderne/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/b8_rl_moderne/03_notebook.ipynb`, créée par `python tools/start_chapter.py B8`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B8.Q1 à B8.Q12, B8.R1 à B8.R3, B8.1 à B8.6 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B8.12 à B8.15 | la valeur : CartPole pour un réseau, ton DQN avec sa mémoire de rejeu et son réseau cible, le DQN qui diverge, les ablations sur trois graines (*prochaine session de génération*) | | | |
| B | B8.16 à B8.24 | la politique : REINFORCE avec et sans baseline, GAE et l'objectif clippé dans ta librairie, ton PPO sur des environnements vectorisés, ses journaux, ses hyperparamètres et sa robustesse (*prochaine session de génération*) | | | |
| C | B8.25 à B8.28 | les préférences : Bradley-Terry et DPO dans ta librairie, un RLHF miniature sur CartPole, la revue de code face à CleanRL, et le défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B8")  # your own library

## Objectifs et rappel express

- Expliquer pourquoi et comment un réseau remplace la table Q, et ce qui rend DQN stable.
- Dériver le gradient de politique et justifier la baseline, l'avantage et GAE.
- Implémenter en PyTorch un DQN, REINFORCE et PPO sur CartPole, avec les calculs clés testés dans `mylearn`.
- Diagnostiquer un entraînement RL à partir de ses journaux et comparer des variantes sur plusieurs graines.
- Expliquer RLHF, DPO et GRPO et entraîner un modèle de récompense sur des préférences.

**Rappel express.** Cible DQN $y = r + \gamma(1 - d)\max_{a'} Q_{\bar\theta}(s', a')$ ; Double DQN : $Q_{\bar\theta}(s', \arg\max_{a'} Q_\theta(s', a'))$. Huber : $\frac12 x^2$ si $|x| \le \delta$, $\delta(|x| - \frac{\delta}{2})$ sinon. $\nabla_{\mathbf z} \log \operatorname{softmax}(\mathbf z)_a = \mathbf e_a - \operatorname{softmax}(\mathbf z)$. $\delta_t = r_t + \gamma(1 - \text{terminated}_t)V(s_{t+1}) - V(s_t)$, $\hat A_t = \delta_t + \gamma\lambda(1 - \text{done}_t)\hat A_{t+1}$. PPO : $-\operatorname{mean}\min(\rho \hat A, \operatorname{clip}(\rho, 1 \pm \varepsilon)\hat A)$. Bradley-Terry : $-\log\sigma(r_w - r_l)$ ; DPO : $-\log\sigma\big(\beta[(\log\pi_w - \log\pi^{\text{ref}}_w) - (\log\pi_l - \log\pi^{\text{ref}}_l)]\big)$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B8.1 à B8.6)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix : les lettres ensemble, entre guillemets, dans n'importe quel ordre (`"EF"`) ; une forme : entre guillemets (`"(3, 5)"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ B8.7, l'estimation 🧮 B8.8, la réflexion (🗣️ B8.9, ⚖️ B8.10, 📄 B8.11) et l'entretien se corrigent avec `05_solutions.md`.

**B8.Q1 — Pourquoi une table Q ne suffit plus pour CartPole**

In [ ]:
# B8.Q1: the values you found on paper
answer_B8_Q1a = ...  # a) a whole number
answer_B8_Q1b = ...  # b) a whole number
answer_B8_Q1c = ...  # c) True or False
answer_B8_Q1d = ...  # d) True or False
answer_B8_Q1e = ...  # e) a shape as a string, e.g. "(3, 5)"

for letter, answer in zip("abcde", [answer_B8_Q1a, answer_B8_Q1b, answer_B8_Q1c, answer_B8_Q1d, answer_B8_Q1e]):
    wb.check(f"B8.Q1{letter}", answer)

**B8.Q2 — La triade mortelle : approximation, amorçage et apprentissage off-policy**

In [ ]:
# B8.Q2: the values you found on paper
answer_B8_Q2a = ...  # a) the letters, e.g. "EF"
answer_B8_Q2b = ...  # b) a whole number
answer_B8_Q2c = ...  # c) a whole number
answer_B8_Q2d = ...  # d) a whole number
answer_B8_Q2e = ...  # e) True or False
answer_B8_Q2f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B8_Q2a, answer_B8_Q2b, answer_B8_Q2c, answer_B8_Q2d, answer_B8_Q2e, answer_B8_Q2f]):
    wb.check(f"B8.Q2{letter}", answer)

**B8.Q3 — Réseau cible et mémoire de rejeu : quel problème chacun résout-il ?**

In [ ]:
# B8.Q3: the values you found on paper
answer_B8_Q3a = ...  # a) a whole number
answer_B8_Q3b = ...  # b) True or False
answer_B8_Q3c = ...  # c) True or False
answer_B8_Q3d = ...  # d) a number, 2 decimals
answer_B8_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B8_Q3a, answer_B8_Q3b, answer_B8_Q3c, answer_B8_Q3d, answer_B8_Q3e]):
    wb.check(f"B8.Q3{letter}", answer)

**B8.Q4 — Double DQN : d'où vient la surestimation des valeurs Q ?**

In [ ]:
# B8.Q4: the values you found on paper
answer_B8_Q4a = ...  # a) the letter of your choice, e.g. "E"
answer_B8_Q4b = ...  # b) a number, 2 decimals
answer_B8_Q4c = ...  # c) a whole number
answer_B8_Q4d = ...  # d) the letter of your choice, e.g. "E"
answer_B8_Q4e = ...  # e) True or False
answer_B8_Q4f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B8_Q4a, answer_B8_Q4b, answer_B8_Q4c, answer_B8_Q4d, answer_B8_Q4e, answer_B8_Q4f]):
    wb.check(f"B8.Q4{letter}", answer)

**B8.Q5 — Méthodes par valeur ou par politique : actions continues et politiques aléatoires**

In [ ]:
# B8.Q5: the values you found on paper
answer_B8_Q5a = ...  # a) the letter of your choice, e.g. "E"
answer_B8_Q5b = ...  # b) a whole number
answer_B8_Q5c = ...  # c) a number, 2 decimals
answer_B8_Q5d = ...  # d) the letters, e.g. "EF"
answer_B8_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B8_Q5a, answer_B8_Q5b, answer_B8_Q5c, answer_B8_Q5d, answer_B8_Q5e]):
    wb.check(f"B8.Q5{letter}", answer)

**B8.Q6 — REINFORCE : pourquoi soustraire une baseline ne biaise pas le gradient**

In [ ]:
# B8.Q6: the values you found on paper
answer_B8_Q6a = ...  # a) a number, 2 decimals
answer_B8_Q6b = ...  # b) a number, 2 decimals
answer_B8_Q6c = ...  # c) the letter of your choice, e.g. "E"
answer_B8_Q6d = ...  # d) True or False
answer_B8_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B8_Q6a, answer_B8_Q6b, answer_B8_Q6c, answer_B8_Q6d, answer_B8_Q6e]):
    wb.check(f"B8.Q6{letter}", answer)

**B8.Q7 — Avantage, erreur TD et GAE : ce que règle λ**

In [ ]:
# B8.Q7: the values you found on paper
answer_B8_Q7a = ...  # a) a number, 2 decimals
answer_B8_Q7b = ...  # b) the letter of your choice, e.g. "E"
answer_B8_Q7c = ...  # c) the letter of your choice, e.g. "E"
answer_B8_Q7d = ...  # d) a number, 2 decimals
answer_B8_Q7e = ...  # e) True or False
answer_B8_Q7f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B8_Q7a, answer_B8_Q7b, answer_B8_Q7c, answer_B8_Q7d, answer_B8_Q7e, answer_B8_Q7f]):
    wb.check(f"B8.Q7{letter}", answer)

**B8.Q8 — PPO : à quoi sert le clip, et pourquoi réutiliser plusieurs fois le même lot ?**

In [ ]:
# B8.Q8: the values you found on paper
answer_B8_Q8a = ...  # a) the letter of your choice, e.g. "E"
answer_B8_Q8b = ...  # b) True or False
answer_B8_Q8c = ...  # c) a whole number
answer_B8_Q8d = ...  # d) a number, 2 decimals
answer_B8_Q8e = ...  # e) True or False
answer_B8_Q8f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B8_Q8a, answer_B8_Q8b, answer_B8_Q8c, answer_B8_Q8d, answer_B8_Q8e, answer_B8_Q8f]):
    wb.check(f"B8.Q8{letter}", answer)

**B8.Q9 — On-policy ou off-policy : DQN, REINFORCE et PPO**

In [ ]:
# B8.Q9: the values you found on paper
answer_B8_Q9a = ...  # a) the letter of your choice, e.g. "E"
answer_B8_Q9b = ...  # b) True or False
answer_B8_Q9c = ...  # c) True or False
answer_B8_Q9d = ...  # d) the letter of your choice, e.g. "E"
answer_B8_Q9e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B8_Q9a, answer_B8_Q9b, answer_B8_Q9c, answer_B8_Q9d, answer_B8_Q9e]):
    wb.check(f"B8.Q9{letter}", answer)

**B8.Q10 — Évaluer un agent : graines, épisodes, politique déterministe**

In [ ]:
# B8.Q10: the values you found on paper
answer_B8_Q10a = ...  # a) a whole number
answer_B8_Q10b = ...  # b) a number, 1 decimal
answer_B8_Q10c = ...  # c) True or False
answer_B8_Q10d = ...  # d) the letter of your choice, e.g. "E"
answer_B8_Q10e = ...  # e) True or False
answer_B8_Q10f = ...  # f) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdef", [answer_B8_Q10a, answer_B8_Q10b, answer_B8_Q10c, answer_B8_Q10d, answer_B8_Q10e, answer_B8_Q10f]):
    wb.check(f"B8.Q10{letter}", answer)

**B8.Q11 — RLHF, DPO, GRPO : qui a besoin d'un modèle de récompense ou d'un critique ?**

In [ ]:
# B8.Q11: the values you found on paper
answer_B8_Q11a = ...  # a) the letters, e.g. "EF"
answer_B8_Q11b = ...  # b) the letters, e.g. "EF"
answer_B8_Q11c = ...  # c) a number, 2 decimals
answer_B8_Q11d = ...  # d) True or False
answer_B8_Q11e = ...  # e) the letter of your choice, e.g. "E"
answer_B8_Q11f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B8_Q11a, answer_B8_Q11b, answer_B8_Q11c, answer_B8_Q11d, answer_B8_Q11e, answer_B8_Q11f]):
    wb.check(f"B8.Q11{letter}", answer)

**B8.Q12 — Piratage de la récompense (reward hacking) : quand l'agent optimise la lettre et pas l'esprit**

In [ ]:
# B8.Q12: the values you found on paper
answer_B8_Q12a = ...  # a) the letters, e.g. "EF"
answer_B8_Q12b = ...  # b) the letter of your choice, e.g. "E"
answer_B8_Q12c = ...  # c) True or False
answer_B8_Q12d = ...  # d) the letter of your choice, e.g. "E"
answer_B8_Q12e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B8_Q12a, answer_B8_Q12b, answer_B8_Q12c, answer_B8_Q12d, answer_B8_Q12e]):
    wb.check(f"B8.Q12{letter}", answer)

**B8.R1 — La mise à jour du Q-learning et le rôle de la mémoire de rejeu**

In [ ]:
# B8.R1: the values you found on paper
answer_B8_R1a = ...  # a) a number, 2 decimals
answer_B8_R1b = ...  # b) a number, 3 decimals
answer_B8_R1c = ...  # c) a number, 2 decimals
answer_B8_R1d = ...  # d) the letter of your choice, e.g. "E"
answer_B8_R1e = ...  # e) True or False
answer_B8_R1f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B8_R1a, answer_B8_R1b, answer_B8_R1c, answer_B8_R1d, answer_B8_R1e, answer_B8_R1f]):
    wb.check(f"B8.R1{letter}", answer)

**B8.R2 — Adam et le gradient clipping**

In [ ]:
# B8.R2: the values you found on paper
answer_B8_R2a = ...  # a) a number, 4 decimals
answer_B8_R2b = ...  # b) a number, 3 decimals
answer_B8_R2c = ...  # c) True or False
answer_B8_R2d = ...  # d) the letter of your choice, e.g. "E"
answer_B8_R2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B8_R2a, answer_B8_R2b, answer_B8_R2c, answer_B8_R2d, answer_B8_R2e]):
    wb.check(f"B8.R2{letter}", answer)

**B8.R3 — Exploration, softmax et bandits**

In [ ]:
# B8.R3: the values you found on paper
answer_B8_R3a = ...  # a) a number, 2 decimals
answer_B8_R3b = ...  # b) a number, 3 decimals
answer_B8_R3c = ...  # c) a number, 1 decimal
answer_B8_R3d = ...  # d) True or False
answer_B8_R3e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B8_R3a, answer_B8_R3b, answer_B8_R3c, answer_B8_R3d, answer_B8_R3e]):
    wb.check(f"B8.R3{letter}", answer)

**Ex B8.1 — Cible DQN à la main : état terminal, puis cible Double DQN**

In [ ]:
# Ex B8.1: the values you found on paper
answer_B8_1a = ...  # a) a number, 2 decimals
answer_B8_1b = ...  # b) a number, 2 decimals
answer_B8_1c = ...  # c) a number, 2 decimals
answer_B8_1d = ...  # d) a number, 2 decimals
answer_B8_1e = ...  # e) a number, 2 decimals
answer_B8_1f = ...  # f) a number, 2 decimals
answer_B8_1g = ...  # g) True or False
answer_B8_1h = ...  # h) a number, 2 decimals

for letter, answer in zip("abcdefgh", [answer_B8_1a, answer_B8_1b, answer_B8_1c, answer_B8_1d, answer_B8_1e, answer_B8_1f, answer_B8_1g, answer_B8_1h]):
    wb.check(f"B8.1{letter}", answer)

**Ex B8.2 — Loss de Huber et son gradient pour trois erreurs TD**

In [ ]:
# Ex B8.2: the values you found on paper
answer_B8_2a = ...  # a) a number, 2 decimals
answer_B8_2b = ...  # b) a number, 3 decimals
answer_B8_2c = ...  # c) a number, 2 decimals
answer_B8_2d = ...  # d) a number, 1 decimal
answer_B8_2e = ...  # e) a number, 2 decimals
answer_B8_2f = ...  # f) a number, 3 decimals
answer_B8_2g = ...  # g) a number, 3 decimals
answer_B8_2h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B8_2a, answer_B8_2b, answer_B8_2c, answer_B8_2d, answer_B8_2e, answer_B8_2f, answer_B8_2g, answer_B8_2h]):
    wb.check(f"B8.2{letter}", answer)

**Ex B8.3 — Gradient du log d'une politique softmax à trois actions**

In [ ]:
# Ex B8.3: the values you found on paper
answer_B8_3a = ...  # a) a number, 4 decimals
answer_B8_3b = ...  # b) a number, 4 decimals
answer_B8_3c = ...  # c) a number, 4 decimals
answer_B8_3d = ...  # d) a number, 4 decimals
answer_B8_3e = ...  # e) True or False
answer_B8_3f = ...  # f) a number, 3 decimals
answer_B8_3g = ...  # g) a number, 4 decimals
answer_B8_3h = ...  # h) a number, 3 decimals

for letter, answer in zip("abcdefgh", [answer_B8_3a, answer_B8_3b, answer_B8_3c, answer_B8_3d, answer_B8_3e, answer_B8_3f, answer_B8_3g, answer_B8_3h]):
    wb.check(f"B8.3{letter}", answer)

**Ex B8.4 — GAE à la main sur quatre pas, pour λ = 0, 0,95 et 1**

In [ ]:
# Ex B8.4: the values you found on paper
answer_B8_4a = ...  # a) a number, 2 decimals
answer_B8_4b = ...  # b) a number, 2 decimals
answer_B8_4c = ...  # c) a number, 2 decimals
answer_B8_4d = ...  # d) a number, 4 decimals
answer_B8_4e = ...  # e) a number, 4 decimals
answer_B8_4f = ...  # f) a number, 4 decimals
answer_B8_4g = ...  # g) a number, 3 decimals
answer_B8_4h = ...  # h) a number, 3 decimals
answer_B8_4i = ...  # i) a number, 3 decimals
answer_B8_4j = ...  # j) True or False

for letter, answer in zip("abcdefghij", [answer_B8_4a, answer_B8_4b, answer_B8_4c, answer_B8_4d, answer_B8_4e, answer_B8_4f, answer_B8_4g, answer_B8_4h, answer_B8_4i, answer_B8_4j]):
    wb.check(f"B8.4{letter}", answer)

**Ex B8.5 — Objectif clippé de PPO : quatre cas, valeur et gradient nul ou non**

In [ ]:
# Ex B8.5: the values you found on paper
answer_B8_5a = ...  # a) a number, 2 decimals
answer_B8_5b = ...  # b) a number, 2 decimals
answer_B8_5c = ...  # c) a number, 2 decimals
answer_B8_5d = ...  # d) a number, 3 decimals
answer_B8_5e = ...  # e) a number, 3 decimals
answer_B8_5f = ...  # f) a number, 2 decimals
answer_B8_5g = ...  # g) a whole number
answer_B8_5h = ...  # h) a number, 2 decimals
answer_B8_5i = ...  # i) a whole number

for letter, answer in zip("abcdefghi", [answer_B8_5a, answer_B8_5b, answer_B8_5c, answer_B8_5d, answer_B8_5e, answer_B8_5f, answer_B8_5g, answer_B8_5h, answer_B8_5i]):
    wb.check(f"B8.5{letter}", answer)

**Ex B8.6 — Préférences : loss de Bradley-Terry puis loss DPO sur une paire de réponses**

In [ ]:
# Ex B8.6: the values you found on paper
answer_B8_6a = ...  # a) a number, 3 decimals
answer_B8_6b = ...  # b) a number, 3 decimals
answer_B8_6c = ...  # c) a number, 3 decimals
answer_B8_6d = ...  # d) a number, 3 decimals
answer_B8_6e = ...  # e) True or False
answer_B8_6f = ...  # f) a number, 2 decimals
answer_B8_6g = ...  # g) a number, 2 decimals
answer_B8_6h = ...  # h) a number, 3 decimals
answer_B8_6i = ...  # i) a number, 3 decimals
answer_B8_6j = ...  # j) a number, 4 decimals

for letter, answer in zip("abcdefghij", [answer_B8_6a, answer_B8_6b, answer_B8_6c, answer_B8_6d, answer_B8_6e, answer_B8_6f, answer_B8_6g, answer_B8_6h, answer_B8_6i, answer_B8_6j]):
    wb.check(f"B8.6{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une cible DQN et Double DQN, en distinguant une vraie fin d'une limite de temps ?
2. Sais-tu dériver le gradient du log d'une politique softmax, et dire pourquoi une baseline ne biaise pas le gradient ?
3. Sais-tu calculer GAE à rebours et l'objectif clippé de PPO avec son gradient, cas par cas ?
4. Sais-tu calculer une loss de Bradley-Terry et une loss DPO, et dire ce que règle $\beta$ ?

**Pour aller plus loin** : l'article de Schulman et coll. (📄 B8.11) et les « 37 détails » de Huang et coll., cités dans la fiche. B8 est le dernier chapitre bonus.

*Les exercices de code B8.12 à B8.28 (la valeur, la politique, les préférences) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B8` pour obtenir le notebook complet.*